# Challenge 2: Spatial Mismatch

| Product | Resolution | Tile Dimensions (pixels) | Per Pixel coverage | Ground coverage |
| :------- | :------: | :------: | :------: | :------: |
| MOD13Q1 | 250 m | 4800 × 4800 | 250 m x 250 m | 1200 km × 1200 km |
| MOD11A2 | 1 km | 1200 × 1200 | 1 km x 1 km | 1200 km × 1200 km |

Match NDVI, EVI from MOD13Q1 with LST from MOD11A2 so that each NDVI, EVI value aligns with its corresponding LST value - one value per grid cell.

**Aggregate MOD13Q1 (250m) to match MOD11A2 (1km) resolution**

* **Original shape:** 4800 × 4800 pixels (each pixel = 250m)
* **Goal:** Match the 1 km spatial resolution of MOD11A2
* **Method:** Aggregate (mean) each 4 × 4 block of 250 m pixels into a single pixel of 1km
* **Output shape:** 1200 × 1200 pixels (each pixel = 1 km)

**Import Packages**

In [8]:
from pyhdf.SD import SD, SDC
import matplotlib.pyplot as plt
import numpy as np 

**Spatial Aggregation Function**

In [9]:
def aggregate_to_1km(data):
    """
    Aggregates 4800x4800 250m data to 1200x1200 1km data
    by taking the mean of each 4x4 block.
    
    Returns:
        np.ndarray: Aggregated 1km image of shape (1200, 1200)
    """
    if data.shape == (4800, 4800):
        # for 4800x4800 data => aggregate 4x4 blocks
        reshaped = data.reshape(1200, 4, 1200, 4)
        aggregated = reshaped.mean(axis=(1, 3))
    elif data.shape == (1200, 1200):
        # No aggregation needed
        aggregated = data
    else:
        raise ValueError(f"Unexpected shape: {data.shape}")
                
    return aggregated # shape: (1200, 1200)

**Process MOD13Q1 - NDVI & EVI (250m)**

In [10]:
def process_mod13q1(file_path):
    """
    Load and process MOD13Q1 NDVI and EVI data from 250m resolution,
    apply scaling, replace fill values, and aggregate to 1km resolution.
    
    Returns:
        Tuple of two NumPy arrays: (ndvi_1km, evi_1km), each shape (1200, 1200)    
    """
    
    # Load the MOD13Q1 tile file
    hdf = SD(file_path, SDC.READ)

    # extract NDVI and EVI 
    # 2D NumPy array of shape (4800, 4800)
    ndvi_data = hdf.select('250m 16 days NDVI')[:]
    evi_data = hdf.select('250m 16 days EVI')[:]

    # Replace fill values (usually -3000)
    # -3000, meaning "no data" or "invalid", replace those with NaN
    ndvi_data = np.where(ndvi_data == -3000, np.nan, ndvi_data)
    evi_data = np.where(evi_data == -3000, np.nan, evi_data)
    
    #print(f"MOD13Q1 NDVI original shape: {ndvi_data.shape}")
    #print(f"MOD13Q1 EVI original shape: {evi_data.shape}")

    # MODIS products like MOD13Q1 store many of their scientific data layers (like NDVI, EVI, reflectance, etc.) as scaled integers in the HDF file.
    # This is done to Save storage (integers are smaller than floats), Maintain precision (avoiding floating-point rounding errors during storage)
    # Apply scale factor (0.0001), converts the raw integer NDVI and EVI values to real physical values in the range -1.0 to 1.0 using the scale factor from the metadata.
    ndvi_scaled = ndvi_data * 0.0001
    evi_scaled = evi_data * 0.0001
    
    # Aggregate to 1km
    ndvi_1km = aggregate_to_1km(ndvi_scaled)
    evi_1km = aggregate_to_1km(evi_scaled)

    hdf.end()
    
    return ndvi_1km, evi_1km